# Locked-Test Evaluation Extensions on v2: Recall at FPR, Intervals, Operating Point

**Run order:** after Phase 2 and Phase 3 have produced `models/phase2-v2-<fp>/` and
`models/phase3-cnn-bigru-v2-<fp>/`. Loads the **saved** models; trains nothing.
**Produces:** `models/eval-v2-<fp>/eval_locked_test.json` and the tables printed below.
**Expect:** a few minutes on CPU (the text model scores 32k emails); no GPU needed.

Covers items 2 and 3 of `docs/PROGRESS.md` Next Actions and `docs/EVALUATION.md` §3.2, §3.4, §4:

1. **Recall at fixed FPR** (1 %, 0.5 %, 0.1 %) on val and on test, for all three v2 models.
2. **Operating points chosen on val** for the two budgets in `docs/OVERVIEW.md` §5 (alert tier
   FPR ≤ 1 %, act tier FPR ≤ 0.1 %), then reported once on test. The F1-max threshold the
   notebooks shipped is reported beside them for comparison, not as the recommendation.
3. **Confidence intervals** on every test number: a row-level percentile bootstrap and a
   **sender-domain-level** bootstrap. The 2026-10-09 slice showed that three held-out bulk-mail
   domains carry 89 % of the text model's test false positives, so the uncertainty lives at the
   domain level and a row-level interval understates it.
4. **Expected calibration error** (10 bins) beside Brier and log loss.
5. Per-corpus slices and the top false-positive sender domains, per model.

Holdout discipline is unchanged: thresholds come from `val`; `test` and `adversarial_test` are
scored once, here, with those thresholds.


## 1. Environment and Artifacts


In [1]:
import json
import os
import platform
import re
from collections import Counter
from contextlib import nullcontext
from datetime import datetime, timezone
from importlib.metadata import version as package_version
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, brier_score_loss, confusion_matrix, f1_score, log_loss, precision_score, recall_score, roc_auc_score
from torch.nn.utils.rnn import pad_sequence
from xgboost import XGBClassifier

SEED = 42
DATASET_VERSION = "v2"


# Resolve the project root so this notebook runs unchanged on the DGX Spark (container or
# host), a Mac, or any other checkout: env var first, then the Spark paths, then walk up
# from the working directory until a folder containing docs/PLAN.md is found.
def resolve_project_dir():
    candidates = [
        Path(p) for p in (
            os.environ.get("EMAIL_NN_DIR"),
            "/workspace/training/email-classifier-nn",
            "/home/spark/projects/training/email-classifier-nn",
        ) if p
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (parent / "docs" / "PLAN.md").exists() and (parent / "notebooks").is_dir():
            return parent
    raise FileNotFoundError(f"project root not found; tried {candidates} and parents of {Path.cwd()}")


def resolve_device():
    # CUDA (DGX Spark) > MPS (Apple Silicon) > CPU. Scoring only; CPU is fine here.
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


PROJECT_DIR = resolve_project_dir()
DATA_DIR = PROJECT_DIR / "data" / f"source-clean-{DATASET_VERSION}"
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
FP = manifest["config_fingerprint"]
FEATURE_COLS = list(manifest["feature_columns"])
PHASE2_DIR = PROJECT_DIR / "models" / f"phase2-{DATASET_VERSION}-{FP[:16]}"
PHASE3_DIR = PROJECT_DIR / "models" / f"phase3-cnn-bigru-{DATASET_VERSION}-{FP[:16]}"
OUTPUT_DIR = PROJECT_DIR / "models" / f"eval-{DATASET_VERSION}-{FP[:16]}"
for path in (PHASE2_DIR, PHASE3_DIR):
    assert path.exists(), f"missing run directory {path}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

frames = {name: pd.read_parquet(DATA_DIR / f"{name}.parquet") for name in ("val", "test", "adversarial_test")}
for name, frame in frames.items():
    expected = manifest["counts"][name]
    assert len(frame) == expected["rows"] and int(frame["label"].sum()) == expected["phish"], name
assert not frames["val"]["augmented"].any() and not frames["test"]["augmented"].any()
assert frames["adversarial_test"]["label"].eq(1).all()


def sender_domain(sender):
    if not isinstance(sender, str):
        return None
    m = re.search(r"@([A-Za-z0-9.-]+)", sender)
    return m.group(1).lower().rstrip(".") if m else None


for frame in frames.values():
    frame["domain"] = frame["sender"].map(sender_domain)
y = {name: frame["label"].to_numpy(dtype=np.int64) for name, frame in frames.items()}

device = resolve_device()
print({"project": str(PROJECT_DIR), "fingerprint": FP[:16], "device": str(device),
       "phase2": PHASE2_DIR.name, "phase3": PHASE3_DIR.name, "output": OUTPUT_DIR.name})
print({name: len(frame) for name, frame in frames.items()})
print("test rows without a parseable sender domain:", int(frames["test"]["domain"].isna().sum()))


{'project': '/workspace/training/email-classifier-nn', 'fingerprint': '72487aa3531fde58', 'device': 'cpu', 'phase2': 'phase2-v2-72487aa3531fde58', 'phase3': 'phase3-cnn-bigru-v2-72487aa3531fde58', 'output': 'eval-v2-72487aa3531fde58'}
{'val': 16116, 'test': 16116, 'adversarial_test': 7465}
test rows without a parseable sender domain: 2703


## 2. Metric Helpers

Same definitions as the Phase 2 notebook, plus recall at fixed FPR, row- and domain-level bootstrap, ECE and slices.


In [2]:
def binary_metrics(y_true, probabilities, threshold):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    predictions = (probabilities >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "average_precision": float(average_precision_score(y_true, probabilities)),
        "brier": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(log_loss(y_true, probabilities, labels=[0, 1])),
        "ece_10_bins": expected_calibration_error(y_true, probabilities),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "false_positive_rate": float(fp / (tn + fp)) if tn + fp else None,
    }


def expected_calibration_error(y_true, probabilities, bins=10):
    y_true = np.asarray(y_true); probabilities = np.asarray(probabilities, dtype=np.float64)
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for low, high in zip(edges[:-1], edges[1:]):
        mask = (probabilities >= low) & (probabilities < high) if high < 1.0 else (probabilities >= low) & (probabilities <= high)
        if mask.any():
            ece += mask.mean() * abs(y_true[mask].mean() - probabilities[mask].mean())
    return float(ece)


FPR_BUDGETS = (0.01, 0.005, 0.001)


def threshold_at_fpr(y_true, probabilities, budget):
    # Lowest threshold whose FPR on these rows is within budget.
    negatives = np.sort(np.asarray(probabilities, dtype=np.float64)[np.asarray(y_true) == 0])
    allowed = int(np.floor(budget * len(negatives)))
    return float(np.nextafter(negatives[-allowed - 1], 1.0)) if allowed < len(negatives) else 0.0


def recall_at_fpr(y_true, probabilities, budgets=FPR_BUDGETS):
    y_true = np.asarray(y_true); probabilities = np.asarray(probabilities, dtype=np.float64)
    table = {}
    for budget in budgets:
        threshold = threshold_at_fpr(y_true, probabilities, budget)
        table[f"fpr_{budget:g}"] = {
            "threshold": threshold,
            "recall": float(np.mean(probabilities[y_true == 1] >= threshold)),
            "achieved_fpr": float(np.mean(probabilities[y_true == 0] >= threshold)),
        }
    return table


def bootstrap_ci(y_true, probabilities, threshold, groups=None, n_resamples=1000, seed=SEED, alpha=0.05):
    # Percentile bootstrap. With `groups` (sender domains), whole groups are resampled with
    # replacement, so the interval reflects which domains happened to land in the split.
    y_true = np.asarray(y_true); probabilities = np.asarray(probabilities, dtype=np.float64)
    predictions = (probabilities >= threshold).astype(np.int64)
    rng = np.random.default_rng(seed)
    n = len(y_true)
    if groups is None:
        draw = lambda: rng.integers(0, n, n)
    else:
        keys = pd.Series(groups).fillna(pd.Series([f"__row{i}" for i in range(n)])).to_numpy()
        members = pd.Series(np.arange(n)).groupby(keys).apply(lambda s: s.to_numpy()).tolist()
        g = len(members)
        draw = lambda: np.concatenate([members[k] for k in rng.integers(0, g, g)])
    samples = {"average_precision": [], "roc_auc": [], "f1": [], "recall": [], "false_positive_rate": []}
    for _ in range(n_resamples):
        index = draw()
        yt, pr, pd_ = y_true[index], probabilities[index], predictions[index]
        if yt.min() == yt.max():
            continue
        tp = int(((pd_ == 1) & (yt == 1)).sum()); fp = int(((pd_ == 1) & (yt == 0)).sum())
        fn = int(((pd_ == 0) & (yt == 1)).sum()); tn = int(((pd_ == 0) & (yt == 0)).sum())
        samples["average_precision"].append(average_precision_score(yt, pr))
        samples["roc_auc"].append(roc_auc_score(yt, pr))
        samples["f1"].append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0)
        samples["recall"].append(tp / (tp + fn) if tp + fn else 0.0)
        samples["false_positive_rate"].append(fp / (fp + tn) if fp + tn else 0.0)
    lo, hi = 100 * alpha / 2, 100 * (1 - alpha / 2)
    return {key: {"low": float(np.percentile(v, lo)), "high": float(np.percentile(v, hi)), "resamples": len(v)} for key, v in samples.items()}


def slice_metrics(frame, probabilities, threshold, column):
    rows = {}
    probabilities = np.asarray(probabilities, dtype=np.float64)
    labels = frame["label"].to_numpy()
    for value, index in frame.groupby(column).indices.items():
        yv, pv = labels[index], probabilities[index]
        pred = (pv >= threshold).astype(int)
        entry = {"rows": int(len(index)), "phish": int(yv.sum()), "legit": int((yv == 0).sum())}
        if yv.sum():
            entry["recall"] = float(pred[yv == 1].mean())
        if (yv == 0).sum():
            entry["false_positive_rate"] = float(pred[yv == 0].mean())
        rows[str(value)] = entry
    return rows


def top_fp_domains(frame, probabilities, threshold, k=10):
    legit = frame[frame["label"] == 0].copy()
    legit["fp"] = (np.asarray(probabilities)[legit.index] >= threshold)
    by = legit.groupby("domain").agg(legit_rows=("fp", "size"), false_positives=("fp", "sum"), corpus=("source", lambda s: s.value_counts().index[0]))
    by = by[by["false_positives"] > 0].sort_values("false_positives", ascending=False).head(k)
    by["false_positives"] = by["false_positives"].astype(int)
    return by


def atomic_json(payload, path):
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str))
    os.replace(tmp, path)


## 3. Load the Saved Models and Score val, test, adversarial_test

Model definitions are copied from the Phase 2 and Phase 3 notebooks so the saved state dicts load unchanged.


In [3]:
# ---- Phase 2: XGBoost and MLP on the 12 engineered features ----
phase2_thresholds = json.loads((PHASE2_DIR / "thresholds.json").read_text())
phase2_config = json.loads((PHASE2_DIR / "config.json").read_text())
assert json.loads((PHASE2_DIR / "feature_columns.json").read_text()) == FEATURE_COLS


def apply_calibrator(calibrator, probabilities):
    clipped = np.clip(probabilities, 1e-7, 1 - 1e-7)
    return calibrator.predict_proba(np.log(clipped / (1 - clipped)).reshape(-1, 1))[:, 1]


xgb_model = XGBClassifier()
xgb_model.load_model(PHASE2_DIR / "xgboost.json")
xgb_model.set_params(device="cpu")
xgb_calibrator = joblib.load(PHASE2_DIR / "xgboost_calibrator.joblib")


class FeatureMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        layers, previous = [], input_dim
        for hidden in hidden_dims:
            layers.extend([nn.Linear(previous, hidden), nn.ReLU(), nn.Dropout(dropout)])
            previous = hidden
        layers.append(nn.Linear(previous, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, features):
        return self.network(features).squeeze(1)


mlp_model = FeatureMLP(len(FEATURE_COLS), phase2_config["mlp"]["hidden_dims"], phase2_config["mlp"]["dropout"])
mlp_model.load_state_dict(torch.load(PHASE2_DIR / "mlp_state.pt", map_location="cpu"))
mlp_model.eval()
mlp_scaler = joblib.load(PHASE2_DIR / "mlp_scaler.joblib")
mlp_calibrator = joblib.load(PHASE2_DIR / "mlp_calibrator.joblib")

# ---- Phase 3: text CNN + BiGRU ----
phase3_metrics = json.loads((PHASE3_DIR / "metrics.json").read_text())
TEXT_CONFIG = phase3_metrics["text_config"]
vocab = json.loads((PHASE3_DIR / "vocab.json").read_text())
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:[._@:/-][a-z0-9]+)*", re.IGNORECASE)
PAD_TOKEN, UNK_TOKEN = "<pad>", "<unk>"


def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())[:TEXT_CONFIG["max_tokens"]]


def encode(text):
    encoded = [vocab.get(token, vocab[UNK_TOKEN]) for token in tokenize(text)]
    return torch.tensor(encoded or [vocab[UNK_TOKEN]], dtype=torch.long)


def email_text(frame):
    return ("subject " + frame["subject"].fillna("").astype(str) + " body " + frame["body"].fillna("").astype(str)).tolist()


class TextCnnBiGru(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config["embedding_dim"], padding_idx=vocab[PAD_TOKEN])
        self.cnn = nn.Conv1d(config["embedding_dim"], config["cnn_channels"], kernel_size=3, padding=1)
        self.gru = nn.GRU(config["cnn_channels"], config["hidden_dim"], batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.Dropout(config["dropout"]), nn.Linear(config["hidden_dim"] * 2, config["hidden_dim"]), nn.ReLU(),
            nn.Dropout(config["dropout"]), nn.Linear(config["hidden_dim"], 1),
        )

    def forward(self, token_ids):
        mask = token_ids.ne(vocab[PAD_TOKEN])
        embedded = self.embedding(token_ids).transpose(1, 2)
        features = torch.relu(self.cnn(embedded)).transpose(1, 2)
        sequence, _ = self.gru(features)
        masked = sequence.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        return self.head(masked.max(dim=1).values).squeeze(1)


cnn_model = TextCnnBiGru(len(vocab), TEXT_CONFIG)
cnn_model.load_state_dict(torch.load(PHASE3_DIR / "model_state.pt", map_location="cpu"))
cnn_model.to(device).eval()


def score_cnn(frame, batch_size=256):
    texts = email_text(frame)
    out = []
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            batch = pad_sequence([encode(t) for t in texts[start:start + batch_size]], batch_first=True, padding_value=vocab[PAD_TOKEN]).to(device)
            out.append(torch.sigmoid(cnn_model(batch).float()).cpu().numpy())
    return np.concatenate(out)


def score_all(frame):
    X = frame[FEATURE_COLS].to_numpy(dtype=np.float32)
    with torch.inference_mode():
        mlp_raw = torch.sigmoid(mlp_model(torch.from_numpy(mlp_scaler.transform(X).astype(np.float32)))).numpy()
    return {
        "xgboost": apply_calibrator(xgb_calibrator, xgb_model.predict_proba(X)[:, 1]),
        "mlp": apply_calibrator(mlp_calibrator, mlp_raw),
        "cnn_bigru": score_cnn(frame),
    }


probabilities = {name: score_all(frame) for name, frame in frames.items()}
shipped_thresholds = {"xgboost": phase2_thresholds["xgboost"], "mlp": phase2_thresholds["mlp"], "cnn_bigru": phase3_metrics["test"]["threshold"]}
MODELS = ["xgboost", "mlp", "cnn_bigru"]

# Reproduce the shipped test numbers before extending them: a mismatch here means a model or
# tokenizer definition drifted from the training notebook.
reference = {"xgboost": json.loads((PHASE2_DIR / "metrics.json").read_text())["test"]["xgboost"],
             "mlp": json.loads((PHASE2_DIR / "metrics.json").read_text())["test"]["mlp"],
             "cnn_bigru": phase3_metrics["test"]}
for name in MODELS:
    here = binary_metrics(y["test"], probabilities["test"][name], shipped_thresholds[name])
    for key in ("recall", "false_positive_rate", "f1"):
        assert abs(here[key] - reference[name][key]) < 0.0015, (name, key, here[key], reference[name][key])
    print(f"{name:10s} reproduces the shipped test metrics: recall={here['recall']:.4f} fpr={here['false_positive_rate']:.4f} f1={here['f1']:.4f}")


xgboost    reproduces the shipped test metrics: recall=0.8086 fpr=0.1611 f1=0.8105
mlp        reproduces the shipped test metrics: recall=0.7976 fpr=0.2737 f1=0.7543
cnn_bigru  reproduces the shipped test metrics: recall=0.9583 fpr=0.1532 f1=0.8974


## 4. Recall at FPR, Operating Points Chosen on val, Intervals, Calibration, Slices

Operating points: for each model, the threshold is the lowest one whose **validation** FPR is
within the budget (1 % alert tier, 0.1 % act tier). That threshold is then applied once to test.
The shipped F1-max threshold is kept in the same table for comparison.


In [4]:
BUDGET_TIERS = {"alert_fpr_1pct": 0.01, "act_fpr_0.1pct": 0.001}
results = {"models": {}, "recall_at_fpr": {"val": {}, "test": {}}}
rows_summary = []
for name in MODELS:
    p_val, p_test, p_adv = probabilities["val"][name], probabilities["test"][name], probabilities["adversarial_test"][name]
    entry = {"operating_points": {}, "shipped_threshold": shipped_thresholds[name]}
    results["recall_at_fpr"]["val"][name] = recall_at_fpr(y["val"], p_val)
    results["recall_at_fpr"]["test"][name] = recall_at_fpr(y["test"], p_test)
    points = {"f1_max_on_val": shipped_thresholds[name]}
    points.update({tier: threshold_at_fpr(y["val"], p_val, budget) for tier, budget in BUDGET_TIERS.items()})
    for point, threshold in points.items():
        test_metrics = binary_metrics(y["test"], p_test, threshold)
        entry["operating_points"][point] = {
            "threshold": threshold,
            "val": binary_metrics(y["val"], p_val, threshold),
            "test": test_metrics,
            "test_ci_rows": bootstrap_ci(y["test"], p_test, threshold),
            "test_ci_domains": bootstrap_ci(y["test"], p_test, threshold, groups=frames["test"]["domain"].to_numpy()),
            "adversarial_recall": float(np.mean(p_adv >= threshold)),
            "test_slices_by_corpus": slice_metrics(frames["test"], p_test, threshold, "source"),
            "test_top_fp_domains": top_fp_domains(frames["test"], p_test, threshold).reset_index().to_dict(orient="records"),
        }
        rows_summary.append({"model": name, "operating_point": point, "threshold": round(threshold, 4),
                             "test_recall": test_metrics["recall"], "test_fpr": test_metrics["false_positive_rate"],
                             "fpr_ci_rows": f"{entry['operating_points'][point]['test_ci_rows']['false_positive_rate']['low']:.4f}-{entry['operating_points'][point]['test_ci_rows']['false_positive_rate']['high']:.4f}",
                             "fpr_ci_domains": f"{entry['operating_points'][point]['test_ci_domains']['false_positive_rate']['low']:.4f}-{entry['operating_points'][point]['test_ci_domains']['false_positive_rate']['high']:.4f}",
                             "recall_ci_domains": f"{entry['operating_points'][point]['test_ci_domains']['recall']['low']:.4f}-{entry['operating_points'][point]['test_ci_domains']['recall']['high']:.4f}",
                             "adv_recall": entry["operating_points"][point]["adversarial_recall"], "ece": test_metrics["ece_10_bins"]})
    results["models"][name] = entry

pd.set_option("display.width", 220)
print("RECALL AT FPR BUDGET (val = selection data; test = locked)")
print(pd.DataFrame({(split, name): {b: round(r["recall"], 4) for b, r in table.items()} for split in ("val", "test") for name, table in results["recall_at_fpr"][split].items()}).T)
print()
print("OPERATING POINTS: threshold chosen on val, reported on test, 95% intervals")
print(pd.DataFrame(rows_summary).to_string(index=False))
print()
for name in MODELS:
    print(f"{name}: top false-positive sender domains at the alert-tier threshold")
    print(pd.DataFrame(results["models"][name]["operating_points"]["alert_fpr_1pct"]["test_top_fp_domains"]).to_string(index=False))
    print()
print("TEST SLICES BY CORPUS at the alert-tier threshold (recall on phishing, FPR on legitimate)")
for name in MODELS:
    s = results["models"][name]["operating_points"]["alert_fpr_1pct"]["test_slices_by_corpus"]
    print(f"  {name:10s}", {k: {m: round(v, 3) for m, v in r.items() if m in ("recall", "false_positive_rate")} for k, r in s.items()})


RECALL AT FPR BUDGET (val = selection data; test = locked)
                fpr_0.01  fpr_0.005  fpr_0.001
val  xgboost      0.4482     0.3587     0.1964
     mlp          0.3408     0.2913     0.1222
     cnn_bigru    0.9309     0.9053     0.7868
test xgboost      0.4249     0.1293     0.0599
     mlp          0.2427     0.1005     0.0473
     cnn_bigru    0.6113     0.5704     0.4213

OPERATING POINTS: threshold chosen on val, reported on test, 95% intervals
    model operating_point  threshold  test_recall  test_fpr   fpr_ci_rows fpr_ci_domains recall_ci_domains  adv_recall      ece
  xgboost   f1_max_on_val     0.4992     0.808573  0.161137 0.1528-0.1684  0.1133-0.2121     0.7718-0.8414    0.999196 0.039267
  xgboost  alert_fpr_1pct     0.8934     0.481581  0.012369 0.0101-0.0147  0.0060-0.0213     0.4325-0.5316    0.988480 0.039267
  xgboost  act_fpr_0.1pct     0.9787     0.248761  0.005780 0.0043-0.0074  0.0004-0.0145     0.2002-0.2968    0.941326 0.039267
      mlp   f1_max_on_va

## 5. Save


In [5]:
results.update({
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_version": DATASET_VERSION, "phase1_config_fingerprint": FP,
    "phase2_run": PHASE2_DIR.name, "phase3_run": PHASE3_DIR.name,
    "budgets": {"recall_at_fpr": list(FPR_BUDGETS), "operating_point_tiers": BUDGET_TIERS, "source": "docs/OVERVIEW.md section 5"},
    "bootstrap": {"resamples": 1000, "row_level": "rows resampled with replacement", "domain_level": "sender domains resampled with replacement; rows without a parseable domain are their own group"},
    "device": str(device), "torch": package_version("torch"), "xgboost": package_version("xgboost"), "python": platform.python_version(),
    "test_domains": int(frames["test"]["domain"].nunique()),
})
atomic_json(results, OUTPUT_DIR / "eval_locked_test.json")
print("written:", OUTPUT_DIR / "eval_locked_test.json")
best = max(MODELS, key=lambda n: results["models"][n]["operating_points"]["alert_fpr_1pct"]["test"]["recall"])
op = results["models"][best]["operating_points"]["alert_fpr_1pct"]
print(f"best at the alert tier (FPR<=1% chosen on val): {best}; test recall {op['test']['recall']:.3f} at test FPR {op['test']['false_positive_rate']:.4f} "
      f"(domain-level 95% FPR interval {op['test_ci_domains']['false_positive_rate']['low']:.4f} to {op['test_ci_domains']['false_positive_rate']['high']:.4f}); target is recall >= 0.95")


written: /workspace/training/email-classifier-nn/models/eval-v2-72487aa3531fde58/eval_locked_test.json
best at the alert tier (FPR<=1% chosen on val): cnn_bigru; test recall 0.939 at test FPR 0.1281 (domain-level 95% FPR interval 0.0138 to 0.2581); target is recall >= 0.95
